# 01 — Build Spatial h5ads

Assembles two spatially-enabled AnnData files from the integrated 602K-cell h5ad:

| Output | Cohort | Cells | Spatial source |
|--------|--------|-------|----------------|
| `outputs/xenium_c1_spatial.h5ad` | Dulai Xenium 5K | 75,389 | `xenium_c1_{1-4}_meta.csv` x/y_centroid (native µm) |
| `outputs/cosmx_c2_spatial.h5ad` | SAHA CosMx | 437,749 | `SAHA_IBD_RNA.h5ad` CenterX/Y_global_px × 0.1203 µm/px |

**Key decisions (from 01-CONTEXT.md):**
- `obsm['spatial']` is always Nx2 float64 in calibrated **microns** (squidpy convention)
- Raw pixel columns preserved in obs as audit trail
- h5py slicing used throughout — never `sc.read_h5ad()` the full 602K file
- SCALE_COSMX = 0.1203 µm/px (empirical, NOT NanoString spec 0.18)

In [ ]:
import h5py
import numpy as np
import scipy.sparse
import anndata
import pandas as pd
import gc
import os
import sys

sys.path.insert(0, "..")  # so 'import config' works from notebooks/
import config

os.makedirs(config.OUTPUT_DIR, exist_ok=True)

INT_PATH = config.INT_H5AD_PATH
print(f"Integration h5ad: {INT_PATH}")
print(f"Exists: {os.path.exists(INT_PATH)}")

## Cell 1 — Xenium c1 assembly (75,389 cells)

In [ ]:
# ── Step 1: Load integrated h5ad obs_names and slice Xenium c1 indices ────────
with h5py.File(INT_PATH, "r") as f:
    raw_names = f["obs"]["_index"][:]
obs_names = [x.decode() if isinstance(x, bytes) else str(x) for x in raw_names]

xen_mask    = np.array([n.startswith("xenium_c1_") for n in obs_names])
xen_indices = np.where(xen_mask)[0]
xen_names   = [obs_names[i] for i in xen_indices]
print(f"Xenium c1 cells: {len(xen_indices)} (expected {config.EXPECTED_XENIUM_C1_CELLS})")
assert len(xen_indices) == config.EXPECTED_XENIUM_C1_CELLS, "Cell count mismatch"

In [ ]:
# ── Step 2: Slice X and embeddings from integrated h5ad ──────────────────────
with h5py.File(INT_PATH, "r") as f:
    # X (CSR sparse matrix)
    data   = f["X"]["data"][:]
    idxs   = f["X"]["indices"][:]
    indptr = f["X"]["indptr"][:]
    shape  = tuple(f["X"].attrs["shape"])
    X_full = scipy.sparse.csr_matrix((data, idxs, indptr), shape=shape)
    X_sub  = X_full[xen_indices, :]

    # Embeddings
    scANVI = f["obsm"]["X_scANVI_deep3"][xen_indices, :]
    scVI   = f["obsm"]["X_scVI_deep3"][xen_indices, :]

    # Obs columns (categoricals require code->category lookup)
    def read_cat(f, key, sel_indices):
        cats  = [x.decode() if isinstance(x, bytes) else str(x)
                 for x in f["obs"][key]["categories"][:]]
        codes = f["obs"][key]["codes"][:]
        return np.array([cats[c] for c in codes])[sel_indices]

    ct_major = read_cat(f, "ct_major", xen_indices)
    sample   = read_cat(f, "sample",   xen_indices)
    dataset  = read_cat(f, "dataset",  xen_indices)

    # var_names (gene names)
    raw_var = f["var"]["_index"][:]
    var_names = [x.decode() if isinstance(x, bytes) else str(x) for x in raw_var]

print(f"X_sub shape: {X_sub.shape}")
print(f"scANVI shape: {scANVI.shape}")
print(f"n_vars: {len(var_names)}")

In [ ]:
# ── Step 3: Join Xenium spatial coordinates from extracted meta CSVs ──────────
# Xenium obs_name format: "xenium_c1_{sample_num}_{barcode}"
# Strip prefix "xenium_c1_{N}_" to recover original barcode (e.g., "aaamakpg-1")

coord_frames = []
for sample_num in range(1, 5):
    csv_path = config.XENIUM_META_CSV[sample_num]
    df = pd.read_csv(csv_path)
    # Ensure barcode column is string, no whitespace
    df["barcode"] = df["barcode"].astype(str).str.strip()
    df["_sample_num"] = sample_num
    coord_frames.append(df[["barcode", "x_centroid", "y_centroid", "_sample_num"]])

coords_all = pd.concat(coord_frames, ignore_index=True)

# Build obs DataFrame with barcode and sample_num extracted from obs_names
obs_df = pd.DataFrame({
    "obs_name": xen_names,
    "ct_major": ct_major,
    "sample":   sample,
    "dataset":  dataset,
})
# Extract sample_num and barcode from obs_name: "xenium_c1_{sample_num}_{barcode}"
obs_df["_sample_num"] = obs_df["obs_name"].str.extract(r"xenium_c1_(\d+)_")[0].astype(int)
obs_df["barcode"]     = obs_df["obs_name"].str.replace(r"^xenium_c1_\d+_", "", regex=True).str.strip()

# Left join on (barcode, _sample_num)
merged = obs_df.merge(
    coords_all,
    on=["barcode", "_sample_num"],
    how="left",
    validate="m:1"  # multiple obs rows can share a sample, but each barcode is unique per sample
)

n_matched = merged["x_centroid"].notna().sum()
match_rate = n_matched / len(merged)
print(f"Xenium c1 coord join: {n_matched}/{len(merged)} = {match_rate*100:.2f}%")
assert match_rate >= config.MIN_MATCH_RATE, f"Match rate {match_rate:.3f} < {config.MIN_MATCH_RATE}"

# Build obsm['spatial'] in native microns (no conversion needed for Xenium)
spatial_um = np.column_stack([
    merged["x_centroid"].values,
    merged["y_centroid"].values
]).astype(np.float64)

In [ ]:
# ── Step 4: Add sample metadata from DULAI_SAMPLE_META in config ─────────────
for col in ["disease", "disease_state", "tissue_type", "block"]:
    merged[col] = merged["_sample_num"].map(
        {k: v[col] for k, v in config.DULAI_SAMPLE_META.items()}
    )

print("Sample disease distribution:")
print(merged.groupby("_sample_num")["disease"].first())

In [ ]:
# ── Step 5: Assemble AnnData and write h5ad ───────────────────────────────────
adata_xen = anndata.AnnData(
    X    = X_sub,
    obs  = merged[["obs_name", "ct_major", "sample", "dataset",
                   "barcode", "disease", "disease_state", "tissue_type", "block"]].set_index("obs_name"),
    var  = pd.DataFrame(index=var_names),
    obsm = {
        "spatial":        spatial_um,
        "X_scANVI_deep3": scANVI,
        "X_scVI_deep3":   scVI,
    }
)

adata_xen.write_h5ad(config.XENIUM_C1_H5AD)
print(f"Written: {config.XENIUM_C1_H5AD}  ({adata_xen.n_obs} cells, {adata_xen.n_vars} genes)")

# Immediate spot-check
adata_check = anndata.read_h5ad(config.XENIUM_C1_H5AD)
assert "spatial" in adata_check.obsm
assert adata_check.obsm["spatial"].shape == (config.EXPECTED_XENIUM_C1_CELLS, 2)
assert "disease" in adata_check.obs.columns
assert "Healthy" in adata_check.obs["disease"].values
print("xenium_c1_spatial.h5ad: spot-check PASSED")

## Cell 2 — SAHA c2 (CosMx c2) assembly (437,749 cells)

In [ ]:
# Free X_full before loading c2 to avoid memory pressure
del X_full
gc.collect()

# ── SAHA c2 (CosMx c2) assembly ───────────────────────────────────────────────
# Cell ID convention: "c2_N" in integrated h5ad = row N in SAHA_IBD_RNA.h5ad (positional, 0-indexed)
# 437,749 cells, all IBD ileum, 8 sections across 2 slides

SAHA_PATH = config.SAHA_H5AD_PATH

# ── Step 1: Get c2 indices from integrated h5ad ───────────────────────────────
# obs_names already loaded above as `obs_names`
c2_mask    = np.array([n.startswith("c2_") for n in obs_names])
c2_indices = np.where(c2_mask)[0]
c2_names   = [obs_names[i] for i in c2_indices]
print(f"SAHA c2 cells: {len(c2_indices)} (expected {config.EXPECTED_COSMX_C2_CELLS})")
assert len(c2_indices) == config.EXPECTED_COSMX_C2_CELLS, "SAHA c2 cell count mismatch"

# Extract row indices in SAHA h5ad: "c2_N" -> N (integer)
saha_row_indices = np.array([int(n[3:]) for n in c2_names])  # strip "c2_" prefix
# Verify: should span 0..437748
assert saha_row_indices.max() == config.EXPECTED_COSMX_C2_CELLS - 1, \
    "SAHA row index max unexpected — check c2_N -> row N mapping"
print(f"SAHA row index range: [{saha_row_indices.min()}, {saha_row_indices.max()}]")

In [ ]:
# ── Step 2: Slice X and embeddings from integrated h5ad ──────────────────────
with h5py.File(INT_PATH, "r") as f:
    # Re-load X_full for c2 slice (X_full was freed above)
    data   = f["X"]["data"][:]
    idxs   = f["X"]["indices"][:]
    indptr = f["X"]["indptr"][:]
    shape  = tuple(f["X"].attrs["shape"])
    X_full_c2 = scipy.sparse.csr_matrix((data, idxs, indptr), shape=shape)
    X_c2      = X_full_c2[c2_indices, :]
    del X_full_c2
    gc.collect()

    scANVI_c2   = f["obsm"]["X_scANVI_deep3"][c2_indices, :]
    scVI_c2     = f["obsm"]["X_scVI_deep3"][c2_indices, :]
    ct_major_c2 = read_cat(f, "ct_major", c2_indices)
    sample_c2   = read_cat(f, "sample",   c2_indices)
    dataset_c2  = read_cat(f, "dataset",  c2_indices)

print(f"X_c2 shape: {X_c2.shape}")
print(f"scANVI_c2 shape: {scANVI_c2.shape}")

In [ ]:
# ── Step 3: Extract spatial coordinates and metadata from SAHA_IBD_RNA.h5ad ──
# CenterX/Y_global_px are integer pixel values in obs.
# obsm['spatial'] in SAHA h5ad also stores raw pixels (NOT microns) — do NOT copy directly.
# Row ordering: obs_names are '0', '1', ..., '437748' (string integers).
# saha_row_indices gives the correct positional row selection.

with h5py.File(SAHA_PATH, "r") as fs:
    # Read pixel coordinates by row index
    cx_px = fs["obs"]["CenterX_global_px"][saha_row_indices]
    cy_px = fs["obs"]["CenterY_global_px"][saha_row_indices]

    # Read SAHA metadata columns
    def read_saha_col(fs, key, sel_indices):
        col = fs["obs"][key]
        # Check if categorical or plain array
        if "categories" in col:
            cats  = [x.decode() if isinstance(x, bytes) else str(x)
                     for x in col["categories"][:]]
            codes = col["codes"][:]
            return np.array([cats[c] for c in codes])[sel_indices]
        else:
            vals = col[:]
            return np.array([v.decode() if isinstance(v, bytes) else v
                             for v in vals])[sel_indices]

    saha_sample = read_saha_col(fs, "SAHA_Sample",             saha_row_indices)
    section_id  = read_saha_col(fs, "section_ID",              saha_row_indices)
    batch       = read_saha_col(fs, "SAHA_withinsample_batch", saha_row_indices)
    fov         = read_saha_col(fs, "fov",                     saha_row_indices)

print(f"cx_px range: [{cx_px.min()}, {cx_px.max()}]")
print(f"cy_px range: [{cy_px.min()}, {cy_px.max()}]")

In [ ]:
# ── Step 4: Apply pixel-to-micron conversion ──────────────────────────────────
# SCALE_COSMX = 0.1203 µm/px (empirically verified from SAHA Area.um2/Area ratio)
spatial_um_c2 = np.column_stack([
    cx_px.astype(np.float64) * config.SCALE_COSMX,
    cy_px.astype(np.float64) * config.SCALE_COSMX,
]).astype(np.float64)

print(f"SAHA c2 spatial range after conversion:")
print(f"  X: [{spatial_um_c2[:,0].min():.1f}, {spatial_um_c2[:,0].max():.1f}] µm")
print(f"  Y: [{spatial_um_c2[:,1].min():.1f}, {spatial_um_c2[:,1].max():.1f}] µm")
# Expect max X ~15,300 µm (NOT ~127,000 pixels)
assert spatial_um_c2[:, 0].max() < 50_000, "Max X > 50,000 µm — pixel conversion likely not applied"

In [ ]:
# ── Step 5: Assemble obs DataFrame ───────────────────────────────────────────
obs_c2 = pd.DataFrame({
    "ct_major":                ct_major_c2,
    "sample":                  sample_c2,
    "dataset":                 dataset_c2,
    "SAHA_Sample":             saha_sample,
    "section_ID":              section_id,
    "SAHA_withinsample_batch": batch,
    "fov":                     fov,
    # Raw pixel coordinates preserved as audit trail (INFRA-06)
    "CenterX_global_px":       cx_px.astype(np.float64),
    "CenterY_global_px":       cy_px.astype(np.float64),
}, index=c2_names)

print(f"obs_c2 shape: {obs_c2.shape}")
print(f"SAHA_Sample unique: {obs_c2['SAHA_Sample'].unique()[:5]}")

In [ ]:
# ── Step 6: Assemble AnnData and write h5ad ───────────────────────────────────
adata_c2 = anndata.AnnData(
    X    = X_c2,
    obs  = obs_c2,
    var  = pd.DataFrame(index=var_names),
    obsm = {
        "spatial":        spatial_um_c2,
        "X_scANVI_deep3": scANVI_c2,
        "X_scVI_deep3":   scVI_c2,
    }
)

adata_c2.write_h5ad(config.COSMX_C2_H5AD)
print(f"Written: {config.COSMX_C2_H5AD}  ({adata_c2.n_obs} cells, {adata_c2.n_vars} genes)")

# Immediate spot-check
adata_c2_check = anndata.read_h5ad(config.COSMX_C2_H5AD)
assert "spatial" in adata_c2_check.obsm
assert adata_c2_check.obsm["spatial"].shape == (config.EXPECTED_COSMX_C2_CELLS, 2)
assert "CenterX_global_px" in adata_c2_check.obs.columns
assert "SAHA_Sample" in adata_c2_check.obs.columns
print("cosmx_c2_spatial.h5ad: spot-check PASSED")